# Figures: pathways and the neuron mix

Reads only `pd-lcm-rf-pathway-composition`. Print size, 183 mm wide, pathway names written in full.

In [ ]:
import os, re, json, glob, textwrap
from pathlib import Path
import numpy as np, pandas as pd
ON_KAGGLE = Path("/kaggle/input").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FIG_OUT", "."))
FIG_IN = Path(os.environ["PWC_IN"]) if not ON_KAGGLE else Path(sorted(
    {str(Path(h).parent) for h in glob.glob("/kaggle/input/**/gprofiler_results.csv", recursive=True)}, key=len)[0])
GPR = pd.read_csv(FIG_IN / "gprofiler_results.csv")
CALR = pd.read_csv(FIG_IN / "gprofiler_random_lists.csv")
SUM = json.load(open(FIG_IN / "pathway_composition_summary.json"))
CAL = SUM["gprofiler"]
LISTLAB = {"up": "Higher in PD (18 genes)", "down": "Lower in PD (12 genes)", "all": "All 30 panel genes"}
SRCLAB = {"GO:BP": "GO BP", "GO:MF": "GO MF", "GO:CC": "GO CC", "REAC": "Reactome", "KEGG": "KEGG", "WP": "WikiPW"}
wrap = lambda t, w: textwrap.wrap(t, w)
def cap(t): return t[0].upper() + t[1:]
print(f"g:Profiler terms: " + ", ".join(f"{k} {int((GPR.list == k).sum()):,} touched, {int(GPR[GPR.list == k].significant.sum())} significant"
                                        for k in ("up", "down", "all")))


In [ ]:
import matplotlib
try:
    get_ipython(); IN_NB = True
except NameError:
    IN_NB = False; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle, Polygon, FancyBboxPatch
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch

INK, MUTED, RULE = "#1B1D20", "#5E656D", "#C9CED4"
PD_C, CT_C = "#7A2533", "#6F829A"
CORE_C, PANEL_C, NEURO_C = "#23324A", "#8E1B2E", "#9AA1A9"
SETS = {"both": ("#8E1B2E", "#5E0F1C", "Boruta & DEG"), "boruta": ("#2E5A87", "#1B3A5C", "Boruta only")}
EFFECT = LinearSegmentedColormap.from_list("effect", ["#1E3350", "#4C6583", "#9AAABB", "#EEECE7", "#C3A09E", "#8C4B52", "#551C28"])
FONT_STACK = ["Helvetica Neue", "Helvetica", "Arial", "Liberation Sans", "Nimbus Sans", "FreeSans", "DejaVu Sans"]
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK, "font.size": 6.3,
                     "axes.linewidth": 0.5, "axes.edgecolor": "#30343A", "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "xtick.labelsize": 5.9, "ytick.labelsize": 5.9,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2.1, "ytick.major.size": 2.1,
                     "xtick.major.pad": 1.7, "ytick.major.pad": 1.7, "xtick.color": "#30343A", "ytick.color": "#30343A",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 150, "savefig.facecolor": "white",
                     "figure.facecolor": "white", "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic"})

class Canvas:
    """A print-size figure drawn on a millimetre grid."""
    def __init__(self, w, h):
        self.W, self.H = w, h
        self.fig = plt.figure(figsize=(w / 25.4, h / 25.4))
        self.M = self.fig.add_axes([0, 0, 1, 1]); self.M.set_xlim(0, w); self.M.set_ylim(0, h); self.M.axis("off")
    def ax(self, x, y, w, h):
        return self.fig.add_axes([x / self.W, y / self.H, w / self.W, h / self.H])
    def letter(self, x, y, L, title):
        self.M.text(x, y, L, ha="left", va="baseline", fontsize=9, fontweight="bold", color=INK)
        self.M.text(x + 4.2, y, title, ha="left", va="baseline", fontsize=7.2, color=INK)
    def rule(self, x0, x1, y, lw=0.4, color=RULE):
        self.M.plot([x0, x1], [y, y], color=color, lw=lw, solid_capstyle="butt")
    def text(self, x, y, s, **kw):
        kw.setdefault("va", "center"); kw.setdefault("fontsize", 6.0)
        return self.M.text(x, y, s, **kw)
    def save(self, stem):
        for ext in ("pdf", "png"):
            self.fig.savefig(OUT / f"{stem}.{ext}", dpi=600 if ext == "png" else None)
        print("saved", stem)
        plt.show() if IN_NB else plt.close(self.fig)
f2 = lambda v: f"{v:.2f}"
def pfmt(p): return f"= {p:.3f}" if p >= 0.001 else "< 0.001"   # "P = 0.003", "P < 0.0001"


## Figure 9 - g:Profiler pathways with the requested settings (183 x 196 mm)

In [ ]:
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch, Circle
import matplotlib.patheffects as pe
UPC, DNC, ALLC = PANEL_C, CORE_C, "#3E454D"
COLS_L = {"up": UPC, "down": DNC, "all": ALLC}
SRC_C = {"GO:BP": "#6B5B7A", "GO:MF": "#4C6583", "GO:CC": "#5F6B3A", "REAC": "#23324A", "KEGG": "#7A2533", "WP": "#8C6B3A"}
RENAME = {"cerebellar climbing fiber to purkinje cell synapse": "Climbing-fibre synapse (GO label)*"}
GPR["disp"] = [RENAME.get(n.lower(), cap(n)) for n in GPR.name]       # the GO label is renamed, not hidden
GPR["fold"] = (GPR.hits / GPR.query_size) / (GPR.term_size / GPR.domain)          # times more panel genes than expected
fmax = max(GPR[GPR.significant].fold.max() if GPR.significant.any() else 1, 1)
GEF = pd.read_csv(FIG_IN / "composition_gene_effects.csv").set_index("symbol")
UPG = set(", ".join(GPR[GPR.list == "up"].genes.dropna()).split(", "))
SIG = GPR[GPR.significant].drop_duplicates("name").sort_values("p_adj")
C = Canvas(183.0, 215.0); M = C.M
N_TOP = {"up": 3, "down": 3, "all": 4}
def chip(x, y_, src):
    M.add_patch(Rectangle((x, y_ - 1.35), 10.4, 2.7, facecolor=SRC_C.get(src, "#E9ECEF"), edgecolor="none", zorder=2))
    C.text(x + 5.2, y_, SRCLAB.get(src, src), ha="center", fontsize=4.6, color="white", zorder=3)

# ======================= a: the results table =======================
C.letter(2.0, 211.0, "a", "g:Profiler with a 5,622-gene background and Benjamini-Hochberg FDR")
SHOW = {k: GPR[GPR.list == k].nsmallest(N_TOP[k], "p_adj") for k in ("up", "down", "all")}
GENES_S = []
for k in ("up", "down", "all"):
    for r in SHOW[k].itertuples():
        for g in str(r.genes).split(", "):
            if g and g not in GENES_S:
                GENES_S.append(g)
GENES_S = sorted(GENES_S, key=lambda g: (g not in UPG, g))
CHX, NX, SZX, FEX, FEW, AX0, AXW, PX, KX, GX0, GX1 = 3.0, 15.0, 74.0, 82.0, 14.0, 100.0, 22.0, 125.0, 134.0, 142.0, 181.0
gcw = (GX1 - GX0) / len(GENES_S)
GXP = {g: GX0 + (i + 0.5) * gcw for i, g in enumerate(GENES_S)}
HY = 198.0
for x, t, ha in ((CHX, "Source", "left"), (NX, "Pathway", "left"), (SZX, "Genes in\nterm / panel", "center"),
                 (FEX + FEW / 2, "Fold\nenrichment", "center"), (AX0 + AXW / 2, "$-\\log_{10}$ FDR", "center"),
                 (PX, "FDR", "center"), (KX, "P vs random\nlists", "center")):
    C.text(x, HY, t, ha=ha, fontsize=5.7, linespacing=1.0)
for g in GENES_S:
    C.text(GXP[g], HY - 1.6, g, ha="center", va="bottom", rotation=90, fontsize=5.1, fontstyle="italic",
           color=UPC if g in UPG else DNC)
C.text((GX0 + GX1) / 2, 209.6, "Panel genes in the pathway", ha="center", fontsize=5.7)
C.rule(2.0, 181.0, HY - 2.6, lw=0.5, color=INK)
ROWS, y_ = [], HY - 6.0
for k in ("up", "down", "all"):
    C.text(NX, y_, LISTLAB[k], ha="left", fontsize=5.8, color=COLS_L[k], fontweight="bold")
    ns = int(GPR[GPR.list == k].significant.sum())
    C.text(CHX, y_, f"{ns} significant", ha="left", fontsize=5.2, color=COLS_L[k] if ns else MUTED)
    y_ -= 4.6
    for r in SHOW[k].itertuples():
        ROWS.append((y_, k, r)); y_ -= 6.2
    y_ -= 1.2
axA = C.ax(AX0, y_ + 3.0, AXW, HY - 3.4 - (y_ + 3.0)); axA.set_ylim(y_ + 3.0, HY - 3.4); axA.set_xlim(0, 3.2)
axA.spines["left"].set_visible(False); axA.set_yticks([]); axA.patch.set_alpha(0)
axA.set_xticks([0, 1, 2, 3]); axA.tick_params(axis="x", labelsize=5.6)
axA.axvline(-np.log10(0.05), color="#8C9299", lw=0.7, ls=(0, (2.2, 1.6)), zorder=1)
for i, (yy, k, r) in enumerate(ROWS):
    col = COLS_L[k]
    if r.significant:
        M.add_patch(Rectangle((2.0, yy - 3.1), 179.0, 6.2, facecolor="#F1F3F6", lw=0, zorder=0))
        M.add_patch(Rectangle((2.0, yy - 3.1), 0.9, 6.2, facecolor=col, lw=0, zorder=1))
    chip(CHX, yy, r.source)
    C.text(NX, yy, "\n".join(wrap(r.disp, 52)), ha="left", fontsize=5.3, linespacing=1.02,
           fontweight="bold" if r.significant else "normal")
    C.text(SZX, yy, f"{int(r.term_size)} / {int(r.hits)}", ha="center", fontsize=5.4)
    fw = FEW * min(r.fold / fmax, 1)
    M.add_patch(Rectangle((FEX, yy - 1.2), max(fw, 0.4), 2.4, facecolor=col if r.significant else "#C9CED4", lw=0))
    C.text(FEX + FEW + 1.0, yy, f"{r.fold:.0f}x", ha="left", fontsize=5.2, color=INK if r.significant else MUTED)
    x = -np.log10(r.p_adj)
    axA.plot([0, x], [yy, yy], color="#C9CED4", lw=0.6, zorder=2)
    axA.scatter([x], [yy], s=8 + 8 * r.hits, color=col if r.significant else "white", edgecolor=col, linewidth=0.7, zorder=3)
    C.text(PX, yy, f"{r.p_adj:.3f}", ha="center", fontsize=5.4, fontweight="bold" if r.significant else "normal")
    C.text(KX, yy, f"{r.calibrated_p:.2f}", ha="center", fontsize=5.4,
           fontweight="bold" if r.calibrated_p < 0.05 else "normal", color=INK if r.calibrated_p < 0.05 else MUTED)
    members = set(str(r.genes).split(", "))
    for g in GENES_S:
        if g in members:
            M.scatter([GXP[g]], [yy], s=13, color=col, lw=0, zorder=3)
        else:
            M.scatter([GXP[g]], [yy], s=1.5, color="#C9CED4", lw=0, zorder=2)
BOT = y_ - 1.0
C.text(3.0, BOT, "Shaded rows pass FDR 0.05. Fold enrichment: panel genes in the term against what chance gives. \"P vs random "
       "lists\": how often 100 random lists of the same size reached a smaller FDR.", ha="left", fontsize=5.3, color=MUTED)
C.text(3.0, BOT - 2.8, "*GO \"cerebellar climbing fibre to Purkinje cell synapse\": a label for a type of excitatory synapse, named "
       "after the tissue where these genes were first described - not a midbrain structure.", ha="left", fontsize=5.3, color=MUTED)
C.rule(2.0, 181.0, BOT - 6.4, lw=0.35, color="#8C9299")

# ======================= b: the graph of significant pathways and their genes =======================
TOPB = BOT - 11.0
C.letter(2.0, TOPB, "b", "Every significant pathway and the panel genes that carry it")
NET = SIG.reset_index(drop=True)
GENES_B = list(dict.fromkeys(sum([str(r.genes).split(", ") for r in NET.itertuples()], [])))
MEMB = {r.disp: str(r.genes).split(", ") for r in NET.itertuples()}
NH = 46.0
axB = C.ax(4.0, TOPB - NH - 2.0, 92.0, NH)
axB.set_xlim(0, 92); axB.set_ylim(0, NH); axB.axis("off"); axB.set_aspect("equal")
cx, cy, R = 44.0, NH / 2, 17.5
ang = {t: np.pi / 2 + 2 * np.pi * i / len(NET) for i, t in enumerate(NET.disp)}
tpos = {t: (cx + R * np.cos(a), cy + R * np.sin(a)) for t, a in ang.items()}
gpos, gdir, used = {}, {}, {}
for g in GENES_B:
    inn = [t for t in NET.disp if g in MEMB[t]]
    if len(inn) > 1:
        bx = np.mean([tpos[t][0] for t in inn]); by = np.mean([tpos[t][1] for t in inn])
        gpos[g] = (cx + (bx - cx) * 0.42, cy + (by - cy) * 0.42)
        v = np.array(gpos[g]) - np.array([cx, cy]); n = np.linalg.norm(v)
        gdir[g] = v / n if n > 1e-6 else np.array([0.0, -1.0])
    else:
        t = inn[0]; k = used.get(t, 0); used[t] = k + 1
        a = ang[t] + (0.32 if k == 0 else -0.32) * (1 if k < 2 else 2)
        gpos[g] = (cx + (R + 11.0) * np.cos(a), cy + (R + 11.0) * np.sin(a))
        gdir[g] = np.array([np.cos(a), np.sin(a)])
SHARED = {}
for g in GENES_B:
    SHARED.setdefault(tuple(sorted(t for t in NET.disp if g in MEMB[t])), []).append(g)
for key, gs in SHARED.items():
    if len(key) > 1 and len(gs) > 1:
        bx, by = gpos[gs[0]]
        for j, g in enumerate(gs):
            a = 2 * np.pi * j / len(gs) + np.pi / 4
            gpos[g] = (bx + 7.5 * np.cos(a), by + 7.5 * np.sin(a))
            gdir[g] = np.array([np.cos(a), np.sin(a)])
STRETCH = 1.9                                        # the panel is twice as wide as it is tall
for d_ in (tpos, gpos):
    for k_ in d_:
        d_[k_] = (cx + (d_[k_][0] - cx) * STRETCH, d_[k_][1])
for g in GENES_B:
    v_ = np.array([gdir[g][0] * STRETCH, gdir[g][1]]); gdir[g] = v_ / np.linalg.norm(v_)
PADX, PADB, PADT = 2.0, 8.0, 1.5                    # room for labels, the legend row, the panel letter
AVW, AVH = 92.0 - 2 * PADX, NH - PADB - PADT
PLBL = 5.9                                           # the "P1" label, in points
RMIN = 0.3528 * PLBL * 0.92                          # a circle this wide holds it with air around it
trad = {r.disp: float(np.sqrt(RMIN ** 2 + 0.085 * min(r.term_size, 60))) for r in NET.itertuples()}
GRAD = 0.95                                          # gene dots
def label_box(g, sc):
    """Where a gene's label lands, in layout units, for the given scale."""
    x, y = gpos[g]; dx, dy = gdir[g]
    ax_, ay_ = x + 3.1 * dx, y + 3.1 * dy
    w, h = 1.18 * len(g) / sc, 2.2 / sc                # the text itself does not shrink with the graph
    lo = ax_ - w / 2 if abs(dx) < 0.45 else (ax_ if dx > 0 else ax_ - w)
    bo = ay_ - h / 2 if abs(dy) < 0.45 else (ay_ if dy > 0 else ay_ - h)
    return lo, lo + w, bo, bo + h
sc = 1.0
for _ in range(8):                                   # label size feeds back into the scale; a few passes settle it
    xs_, ys_ = [], []
    for t_, pnt in tpos.items():
        m_ = (trad[t_] + 0.6) / sc
        xs_ += [pnt[0] - m_, pnt[0] + m_]; ys_ += [pnt[1] - m_, pnt[1] + m_]
    for pnt in gpos.values():
        m_ = (GRAD + 0.4) / sc
        xs_ += [pnt[0] - m_, pnt[0] + m_]; ys_ += [pnt[1] - m_, pnt[1] + m_]
    for g in GENES_B:
        a_, b_, c_, d_ = label_box(g, sc); xs_ += [a_, b_]; ys_ += [c_, d_]
    x_lo, x_hi, y_lo, y_hi = min(xs_), max(xs_), min(ys_), max(ys_)
    sc = min(AVW / (x_hi - x_lo), AVH / (y_hi - y_lo))
ox = PADX + (AVW - sc * (x_hi - x_lo)) / 2 - sc * x_lo
oy = PADB + (AVH - sc * (y_hi - y_lo)) / 2 - sc * y_lo
fit = lambda p_: (ox + sc * p_[0], oy + sc * p_[1])
tpos = {t: fit(p_) for t, p_ in tpos.items()}
gpos = {g: fit(p_) for g, p_ in gpos.items()}
for t in NET.disp:
    for g in MEMB[t]:
        col = UPC if g in UPG else DNC
        (x0, y0), (x1, y1) = tpos[t], gpos[g]
        mx, my = (x0 + x1) / 2 + (y1 - y0) * 0.10, (y0 + y1) / 2 - (x1 - x0) * 0.10
        axB.add_patch(PathPatch(MPath([(x0, y0), (mx, my), (x1, y1)], [MPath.MOVETO, MPath.CURVE3, MPath.CURVE3]),
                                facecolor="none", edgecolor=col, lw=1.1, alpha=0.5, zorder=2))
for i, r in enumerate(NET.itertuples()):
    x, y0 = tpos[r.disp]
    axB.add_patch(Circle((x, y0), trad[r.disp], facecolor="white", edgecolor=SRC_C.get(r.source, INK), linewidth=1.3, zorder=4))
    axB.text(x, y0, f"P{i + 1}", ha="center", va="center", fontsize=PLBL, fontweight="bold", color=SRC_C.get(r.source, INK), zorder=6)
HALO = [pe.withStroke(linewidth=1.8, foreground="white")]
for g in GENES_B:
    x, y0 = gpos[g]; col = UPC if g in UPG else DNC
    dx, dy = gdir[g]
    axB.add_patch(Circle((x, y0), GRAD, facecolor=col, edgecolor="white", linewidth=0.8, zorder=5))
    ha = "center" if abs(dx) < 0.45 else ("left" if dx > 0 else "right")
    va = "center" if abs(dy) < 0.45 else ("bottom" if dy > 0 else "top")
    axB.text(x + (GRAD + 1.3) * dx, y0 + (GRAD + 1.3) * dy, g, ha=ha, va=va, fontsize=5.5, fontstyle="italic", color=col, zorder=6,
             path_effects=HALO)
axB.add_patch(Circle((3.0, 3.4), GRAD, facecolor=UPC, edgecolor="white", linewidth=0.8))
axB.text(5.6, 3.4, "higher in PD", ha="left", va="center", fontsize=5.2, color=MUTED)
axB.add_patch(Circle((28.0, 3.4), GRAD, facecolor=DNC, edgecolor="white", linewidth=0.8))
axB.text(30.6, 3.4, "lower in PD", ha="left", va="center", fontsize=5.2, color=MUTED)
axB.add_patch(Circle((52.0, 3.4), max(trad.values()), facecolor="white", edgecolor="#8C9299", linewidth=1.1))
axB.text(55.6, 3.4, "pathway, ringed by source; area = genes in it", ha="left", va="center", fontsize=5.2, color=MUTED)
LX = 100.0
ly = TOPB - 5.0
for i, r in enumerate(NET.itertuples()):
    C.text(LX, ly, f"P{i + 1}", ha="left", fontsize=5.8, fontweight="bold", color=SRC_C.get(r.source, INK))
    chip(LX + 6.0, ly, r.source)
    C.text(LX + 18.0, ly, f"FDR {r.p_adj:.3f}    {r.fold:.0f}x    {int(r.term_size)} genes    P vs random {r.calibrated_p:.2f}",
           ha="left", fontsize=5.0, color=MUTED)
    C.text(LX, ly - 3.4, "\n".join(wrap(r.disp, 52)), ha="left", va="top", fontsize=5.3, linespacing=1.15)
    ly -= 4.0 + 2.9 * len(wrap(r.disp, 52)) + 1.4
BB = TOPB - NH - 6.0
C.text(2.0, BB, "Four pathways rest on five genes, three of them shared. Those genes are largely markers of which dopamine neurons "
       "survive,", ha="left", fontsize=5.3, color=MUTED)
C.text(2.0, BB - 2.8, "so part of this signal is the neuron mix rather than a pathway (Figure 10).", ha="left", fontsize=5.3, color=MUTED)
C.rule(2.0, 181.0, BB - 6.6, lw=0.35, color="#8C9299")

# ======================= c: what the same settings give on random gene lists =======================
DY = BB - 11.4
C.letter(2.0, DY, "c", "The same settings on 100 random gene lists of the same size")
BW, BH0, BY = 50.0, 17.0, 13.0
for i, k in enumerate(("up", "down", "all")):
    x0 = 16.0 + i * (BW + 6.0)
    ax = C.ax(x0, BY, BW, BH0)
    c = CALR[CALR.list == k].n_significant.to_numpy()
    obs = int(GPR[GPR.list == k].significant.sum())
    top = max(c.max(), obs) + 1
    ax.hist(c, bins=np.arange(-0.5, top + 1.5, 1), color="#C9CED4", lw=0)
    ax.axvline(obs, color=COLS_L[k], lw=1.3, zorder=3)
    ax.text(obs + 0.6, ax.get_ylim()[1] * 0.94, f"the panel: {obs}", fontsize=5.3, color=COLS_L[k], ha="left", va="top")
    ax.set_xlabel("\"Significant\" pathways returned", fontsize=5.6, labelpad=1.4)
    if i == 0:
        ax.set_ylabel("Random lists", fontsize=5.6, labelpad=2)
    ax.tick_params(labelsize=5.3)
    C.text(x0 + BW / 2, BY + BH0 + 1.8, LISTLAB[k], ha="center", fontsize=5.8, color=COLS_L[k], fontweight="bold")
    C.text(x0 + BW / 2, BY - 6.6, f"{100 * CAL[k]['random_with_any_significant']:.0f}% of random lists return at least one "
           f"(mean {CAL[k]['random_mean_significant']:.1f});\nstrongest term vs random lists: P = {CAL[k]['calibrated_p_best']:.2f}",
           ha="center", fontsize=5.3, color=MUTED, linespacing=1.2)
C.save("Figure09_gprofiler_pathways")


## Figure 10 - the neuron-mix test (183 x 165 mm)

In [ ]:
UPC, DNC = PANEL_C, CORE_C
CTC, PDC = CT_C, PD_C
C = Canvas(183.0, 215.0); M = C.M
CO = pd.read_csv(FIG_IN / "composition_scores.csv")
CVT = pd.read_csv(FIG_IN / "composition_cv.csv")
AUCT = pd.read_csv(FIG_IN / "composition_oof_auc.csv")
GEF = pd.read_csv(FIG_IN / "composition_gene_effects.csv")
CSU = json.load(open(FIG_IN / "pathway_composition_summary.json"))
SUMC, LRC = CSU["composition"], CSU["logistic"]
STUDIES = sorted(CO.dataset.unique())

# ======================= a: the neuron mix of every donor =======================
C.letter(2.0, 211.0, "a", "Estimated dopamine-neuron mix of every donor")
axA = C.ax(16.0, 162.0, 74.0, 40.0)
xs = {d: i for i, d in enumerate(STUDIES)}
rngj = np.random.default_rng(3)
for d in STUDIES:
    for lab, col, off in ((0, CTC, -0.17), (1, PDC, 0.17)):
        v = CO[(CO.dataset == d) & (CO.y == lab)].composition_score.to_numpy()
        x = xs[d] + off + rngj.uniform(-0.07, 0.07, len(v))
        axA.scatter(x, v, s=11, color=col, edgecolor="white", linewidth=0.3, zorder=3)
        axA.plot([xs[d] + off - 0.13, xs[d] + off + 0.13], [v.mean()] * 2, color=col, lw=1.1, zorder=4)
axA.axhline(0, color="#C9CED4", lw=0.5, zorder=1)
axA.set_xticks(range(len(STUDIES))); axA.set_xticklabels(STUDIES, fontsize=5.5)
axA.tick_params(axis="x", length=2.0, pad=1.6)
axA.set_xlim(-0.5, len(STUDIES) - 0.5)
axA.set_ylabel("Composition score\n(CALB1-like $-$ SOX6-like)", fontsize=6.0, labelpad=2, linespacing=1.05)
for x, col, t in ((16.0, CTC, "control"), (30.0, PDC, "PD")):
    M.scatter([x], [206.5], s=12, color=col, edgecolor="white", linewidth=0.3); C.text(x + 1.6, 206.5, t, ha="left", fontsize=5.8)
C.text(16.0, 157.0, f"PD donors are more CALB1-like: AUC {SUMC['auc_composition_pd_more_CALB1_like']:.2f}, Mann-Whitney "
       f"P = {SUMC['composition_mwu_p']:.1e}", ha="left", fontsize=5.6, color=INK)
C.text(16.0, 153.6, f"Markers: the {len(CSU['markers']['calb1'])} strongest CALB1-lineage and {len(CSU['markers']['sox6'])} strongest "
       "SOX6-lineage genes of Kamath et al. (2022), panel genes excluded", ha="left", fontsize=5.3, color=MUTED)

# ======================= b: how much the classifier follows the mix =======================
C.letter(100.0, 211.0, "b", "Classifier score against the mix")
axB = C.ax(114.0, 162.0, 62.0, 40.0)
for lab, col, t in ((0, CTC, "control"), (1, PDC, "PD")):
    s_ = CO[CO.y == lab]
    axB.scatter(s_.composition_score, s_.core_oof_score, s=12, color=col, edgecolor="white", linewidth=0.3, zorder=3)
xx = np.linspace(CO.composition_score.min(), CO.composition_score.max(), 50)
axB.plot(xx, np.polyval(np.polyfit(CO.composition_score, CO.core_oof_score, 1), xx), color="#8C9299", lw=0.7, ls=(0, (3, 1.6)), zorder=2)
axB.set_xlabel("Composition score", fontsize=6.0, labelpad=2)
axB.set_ylabel("Core classifier, out-of-fold score", fontsize=6.0, labelpad=2)
axB.text(0.03, 0.97, f"Spearman ρ = {SUMC['rho_oof_composition']:.2f}\ncontrols only ρ = {SUMC['rho_oof_composition_controls']:.2f}",
         transform=axB.transAxes, ha="left", va="top", fontsize=5.6, color=MUTED, linespacing=1.15)
C.rule(2.0, 181.0, 149.0, lw=0.35, color="#8C9299")

# ======================= c: does the classifier survive the mix being removed? =======================
C.letter(2.0, 144.5, "c", "The classifier with the neuron mix regressed out of every gene")
ORDER = ["original", "composition (50 markers)", "composition (25 markers)", "composition (100 markers)", "composition (200 markers)",
         "SOX6_AGTR1 markers only", "composition + dopamine purity"]
LBL = {"original": "No adjustment", "composition (50 markers)": "Composition removed (50 markers)",
       "composition (25 markers)": "25 markers", "composition (100 markers)": "100 markers", "composition (200 markers)": "200 markers",
       "SOX6_AGTR1 markers only": "SOX6/AGTR1 markers only", "composition + dopamine purity": "Composition + dopamine purity"}
core_auc = AUCT[AUCT.model == "core"].set_index("variant")
AXC0, AXCW = 68.0, 52.0
y0, step = 137.0, 5.6
axC = C.ax(AXC0, y0 - (len(ORDER) - 1) * step - 3.0, AXCW, (len(ORDER) - 1) * step + 6.0)
axC.set_ylim(y0 - (len(ORDER) - 1) * step - 3.0, y0 + 3.0); axC.set_xlim(0.4, 1.0)
axC.spines["left"].set_visible(False); axC.set_yticks([]); axC.patch.set_alpha(0)
axC.set_xticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0]); axC.tick_params(axis="x", labelsize=5.6)
axC.axvline(0.5, color="#A9AFB6", lw=0.5, ls=(0, (2.2, 1.8)), zorder=0)
for i, v in enumerate(ORDER):
    yy = y0 - i * step
    r = core_auc.loc[v]
    prim = v in ("original", "composition (50 markers)")
    col = INK if v == "original" else (CORE_C if prim else "#6B7178")
    C.text(AXC0 - 2.0, yy, LBL[v], ha="right", fontsize=5.8, fontweight="bold" if prim else "normal",
           color=INK if prim else MUTED)
    axC.plot([r.ci_lo, r.ci_hi], [yy, yy], color=col, lw=0.8, zorder=2)
    axC.scatter([r.oof_auc], [yy], s=22 if prim else 14, color=col, zorder=3, linewidth=0)
    C.text(AXC0 + AXCW + 2.0, yy, f"{r.oof_auc:.2f} [{r.ci_lo:.2f}–{r.ci_hi:.2f}]", ha="left", fontsize=5.6,
           fontweight="bold" if prim else "normal")
    C.text(AXC0 + AXCW + 28.0, yy, ("P < 0.0001" if r.shuffle_p < 0.0001 else f"P = {r.shuffle_p:.4f}"), ha="left", fontsize=5.6,
           color=INK if r.shuffle_p < 0.05 else MUTED)
C.text(AXC0 + AXCW / 2, y0 + 5.6, "Out-of-fold AUC of the core classifier [95% CI]", ha="center", fontsize=5.8)
C.text(AXC0 + AXCW + 2.0, y0 + 5.6, "AUC [95% CI]", ha="left", fontsize=5.5, color=MUTED)
C.text(AXC0 + AXCW + 28.0, y0 + 5.6, "Label shuffles", ha="left", fontsize=5.5, color=MUTED)
ybot = y0 - (len(ORDER) - 1) * step - 9.0        # clear of the axis tick labels
C.text(2.0, ybot, f"Same 5 x 5 folds and the same model as the paper's classifier. Adding the classifier's score to the composition "
       f"raises the leave-one-out AUC from {LRC['loo_auc_composition']:.2f} to {LRC['loo_auc_composition_plus_score']:.2f}"
       + (f" (likelihood-ratio P = {LRC['lr_p']:.4f})." if "lr_p" in LRC else "."), ha="left", fontsize=5.4, color=MUTED)
C.rule(2.0, 181.0, ybot - 4.0, lw=0.35, color="#8C9299")

# ======================= d: gene by gene =======================
C.letter(2.0, ybot - 8.5, "d", "PD effect of each panel gene before and after the mix is removed")
G = GEF.sort_values("g_original").reset_index(drop=True)
DX0, DW, DY0 = 44.0, 100.0, 12.0
DH = ybot - 12.0 - DY0
axD = C.ax(DX0, DY0, DW, DH)
lim = 1.45
ylo_, yhi_ = -0.8, len(G) - 0.2
axD.set_xlim(-lim, lim); axD.set_ylim(ylo_, yhi_)
row_mm = lambda i: DY0 + DH * (i - ylo_) / (yhi_ - ylo_)
axD.spines["left"].set_visible(False); axD.set_yticks([]); axD.patch.set_alpha(0)
axD.axvline(0, color="#A9AFB6", lw=0.5, zorder=1)
axD.set_xlabel("PD effect in the discovery neurons (Hedges' g)", fontsize=6.0, labelpad=2)
axD.tick_params(axis="x", labelsize=5.6)
for i, r in enumerate(G.itertuples()):
    col = UPC if r.direction == "up" else DNC
    axD.plot([r.g_original, r.g_composition_removed], [i, i], color="#C9CED4", lw=0.8, zorder=2)
    axD.scatter([r.g_original], [i], s=13, facecolor="white", edgecolor=col, linewidth=0.8, zorder=3)
    axD.scatter([r.g_composition_removed], [i], s=13, color=col, zorder=4, linewidth=0)
    yy = row_mm(i)
    C.text(DX0 - 2.0, yy, r.symbol, ha="right", fontsize=5.3, fontstyle="italic", color=col)
    C.text(DX0 + DW + 2.0, yy, f"{100 * r.retained:.0f}%", ha="left", fontsize=5.3,
           color=INK if r.retained >= 0.5 else MUTED)
    C.text(DX0 + DW + 12.0, yy, f"{r.lineage:+.1f}" if abs(r.lineage) >= 0.05 else "0", ha="left", fontsize=5.3, color=MUTED)
C.text(DX0 + DW + 2.0, DY0 + DH + 2.2, "Effect\nkept", ha="left", fontsize=5.4, linespacing=1.0)
C.text(DX0 + DW + 12.0, DY0 + DH + 2.2, "Lineage\nscore", ha="left", fontsize=5.4, color=MUTED, linespacing=1.0)
kx = 4.0
M.scatter([kx], [DY0 + 6.0], s=13, facecolor="white", edgecolor="#6B7178", linewidth=0.8)
C.text(kx + 2.0, DY0 + 6.0, "before", ha="left", fontsize=5.4)
M.scatter([kx], [DY0 + 2.6], s=13, color="#6B7178", linewidth=0)
C.text(kx + 2.0, DY0 + 2.6, "after the mix is removed", ha="left", fontsize=5.4)
C.text(2.0, 4.0, f"{int((GEF.retained >= 0.5).sum())} of 30 genes keep at least half of their PD effect; median kept "
       f"{100 * GEF.retained.median():.0f}%.", ha="left", fontsize=5.4, color=MUTED)
C.save("Figure10_neuron_mix")


## Ready-to-paste legends, methods and numbers

In [ ]:
co = SUM["composition"]; lr = SUM["logistic"]
auc = pd.DataFrame(SUM["oof_auc"]); cv = pd.DataFrame(SUM["cv"])
core = auc[auc.model == "core"].set_index("variant")
o, a = core.loc["original"], core.loc["composition (50 markers)"]
ge = pd.read_csv(FIG_IN / "composition_gene_effects.csv")
pf = lambda p: f"P = {p:.4f}" if p >= 0.0001 else "P < 0.0001"
TEXT = f"""FIGURE 9 LEGEND - Pathway over-representation with g:Profiler
(a) g:GOSt run with the statistical domain set to the {5622:,} genes measured in the discovery data and Benjamini-Hochberg FDR as
the significance threshold, for the genes higher in PD (18), lower in PD (12) and all 30; the six smallest FDR values per list,
with the panel genes behind each pathway. Dots are filled at FDR < 0.05; dot area is the number of panel genes. "P vs random
lists" is how often 100 random gene lists of the same size reached a smaller FDR anywhere under identical settings. (b) The
number of "significant" pathways those random lists returned, with the panel's count marked. Terms holding only 2-4 background
genes reach very small FDR values when both of their genes are in the panel and name the genes rather than a pathway.

FIGURE 10 LEGEND - Does the classifier see more than the neuron mix?
(a) Composition score of every donor: mean z-score of the {len(SUM['markers']['calb1'])} strongest CALB1-lineage markers minus the {len(SUM['markers']['sox6'])} strongest SOX6-lineage
markers of Kamath et al. (2022), computed within each study with the 30 panel genes excluded. PD donors are more CALB1-like
(AUC {co['auc_composition_pd_more_CALB1_like']:.2f}, Mann-Whitney {pf(co['composition_mwu_p'])}), as expected when SOX6/AGTR1 neurons are lost. (b) The classifier's out-of-fold
score against the same composition (Spearman {co['rho_oof_composition']:.2f}; {co['rho_oof_composition_controls']:.2f} within controls alone). (c) Out-of-fold AUC of the core classifier on
the same folds after the composition is regressed out of every gene, with 95% bootstrap CI and P from label shuffles within
study; the marker count and the alternatives test how the estimate depends on that choice. (d) PD effect of each panel gene
before (open) and after (filled) the composition is removed, with the share of the effect kept and the gene's lineage score.

METHODS - Neuron-mix (composition) analysis
Kamath et al. (2022) report markers of ten human dopamine-neuron subtypes; the SOX6 subtypes, in particular SOX6_AGTR1, are the
ones lost in PD, and CALB1 subtypes are relatively spared. Each gene was given a lineage score (mean marker z across the six
CALB1 subtypes minus the mean across the four SOX6 subtypes). A donor's composition score is the mean expression z-score of the
50 strongest CALB1-lineage genes minus that of the 50 strongest SOX6-lineage genes, computed within each study; the 30 panel
genes were excluded, and no diagnosis labels are used. The composition was then regressed out of every gene within each study,
and the core classifier (within-person ranks, 30 principal components, Random Forest) was re-run on exactly the folds of the
main analysis. Significance of the adjusted out-of-fold AUC comes from 10,000 label shuffles within study, its interval from
4,000 bootstrap resamples of donors. Sensitivity analyses used 25, 100 and 200 markers, the SOX6_AGTR1 markers alone, and the
composition together with a dopamine-neuron purity score. Per gene, the pooled within-study PD effect (Hedges' g) was compared
before and after adjustment.

RESULTS - numbers
Composition differs by diagnosis: PD donors are more CALB1-like, AUC {co['auc_composition_pd_more_CALB1_like']:.2f} ({pf(co['composition_mwu_p'])}); SOX6/AGTR1 markers alone {co['auc_AGTR1_markers_pd_lower']:.2f}.
The classifier partly follows it: Spearman {co['rho_oof_composition']:.2f} overall, {co['rho_oof_composition_controls']:.2f} within controls.
With the composition regressed out of every gene, the core classifier keeps a significant part of its accuracy:
  out-of-fold AUC {o.oof_auc:.2f} [{o.ci_lo:.2f}-{o.ci_hi:.2f}] before, {a.oof_auc:.2f} [{a.ci_lo:.2f}-{a.ci_hi:.2f}] after ({pf(a.shuffle_p)}); cross-validated AUC {cv[(cv.variant == 'original') & (cv.model == 'core classifier')].cv_auc.iloc[0]:.3f} -> {cv[(cv.variant == 'composition (50 markers)') & (cv.model == 'core classifier')].cv_auc.iloc[0]:.3f}.
  Sensitivity (out-of-fold AUC, label-shuffle P): """ + "; ".join(f"{v} {core.loc[v].oof_auc:.2f} ({pf(core.loc[v].shuffle_p)})"
        for v in ["composition (25 markers)", "composition (100 markers)", "composition (200 markers)", "SOX6_AGTR1 markers only",
                  "composition + dopamine purity"]) + f"""
  The classifier's score adds to the composition: leave-one-out AUC {lr['loo_auc_composition']:.2f} -> {lr['loo_auc_composition_plus_score']:.2f}""" + (
    f", likelihood-ratio {pf(lr['lr_p'])}, score coefficient {pf(lr['score_coef_p'])}." if "lr_p" in lr else ".") + f"""
  Genes: {int((ge.retained >= 0.5).sum())}/30 keep at least half of their PD effect (median {100 * ge.retained.median():.0f}%). Most composition-dependent: """ + ", ".join(
    f"{r.symbol} {100 * r.retained:.0f}%" for r in ge.nsmallest(5, "retained").itertuples()) + "; least: " + ", ".join(
    f"{r.symbol} {100 * r.retained:.0f}%" for r in ge.nlargest(5, "retained").itertuples()) + """.
CONCLUSION: the panel reflects both which dopamine neurons survive and changes inside the surviving neurons - the classifier
stays above chance after the neuron mix is removed, and most panel genes keep most of their PD effect.
"""
print(TEXT)
open(OUT / "pathway_composition_legends.txt", "w").write(TEXT)
